In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Read in clean BBHI senior data
bbhi_senior = pd.read_csv("~/Documents/2023:2024/Data/Exported data/clean_bbhi_senior_tp1.csv")
bbhi_senior = bbhi_senior.apply(lambda col: col.astype(float) if col.name not in ["id", "sex"] else col)

print(f"Number of BBHI senior participants: {len(bbhi_senior)}")

# Read BBHI education data
bbhi = pd.read_csv("~/Documents/2023:2024/Data/Exported data/clean_bbhi_tp1.csv")
bbhi = bbhi.apply(lambda col: col.astype(float) if col.name not in ["id", "sex"] else col)

print(f"Number of BBHI participants: {len(bbhi)}")

# Identify common columns to merge on
common_columns = [col for col in bbhi.columns if col in bbhi_senior.columns]

# Merge on all common columns
data = pd.merge(bbhi, bbhi_senior, on=common_columns, how="outer")

# Print the number of participants after filtering
print(f"Number of participants: {len(data)}")

data.sample(5)

In [ ]:
# Look at data for available cross-sectional analysis
participants = data["w1_age"].count()
mean_age = data["w1_age"].mean()
std_dev_age = data["w1_age"].std()
min_age = data["w1_age"].min()
max_age = data["w1_age"].max()

print(f"Number of participants: {participants}")
print(f"Mean age: {mean_age:.2f}")
print(f"SD age: {std_dev_age:.2f}")
print(f"Range age: {min_age:.2f} - {max_age:.2f}")

The following is a calculation based on the criteria proposed by [Sun et al. (2016)](https://pubmed.ncbi.nlm.nih.gov/27629716/) on who can be included in a superager analyses and who is a superager. Our criteria are a slight variation as follows:

- All participants must:
  - Be age 60+
  - Have an MMSE score >= 27 (I suggest that this allows us to consider them as healthy agers, rather than the stricter Sun et al. criteria)
- Superagers must:
  - Score at or above the mean for age 16-29 year olds on the RAVLT long delay free recall based on normative data from [Schmidt (1996)](https://scholar.google.co.uk/scholar?hl=en&as_sdt=0%2C5&q=Schmidt%2C+M.+%281996%29.+Rey+Auditory+and+Verbal+Learning+Test%3A+A+handbook.+Los+Angeles%2C+CA%3A+Western+Psychological+Services&btnG=).
  - Score above 1 SD below the norm for age and education on the TMT B based on the neuronorma data from [Peña-Casanova et al. (2009a)](https://pubmed.ncbi.nlm.nih.gov/19661109/) with Spanish adults.

In [ ]:
# Readin the Neuronorma data (from the two publication above) in excel form & put into a df
xls = pd.ExcelFile("/Users/rachelmorse/superagers/Classification/neuronorma_neuropsych_data.xlsx")
score_mappings = {sheet_name: pd.read_excel(xls, sheet_name) for sheet_name in xls.sheet_names}

In [ ]:
# Round down years of education data because some participants have data that is not a whole number (e.g. YoE = 9.5)
# This gives participants the lower education value because they will have their cognitive test scores normalized according to education level, disadvantaging them if it is rounded up
data["YoE"] = np.floor(data["YoE"])

In [ ]:
# Round the ages of the participants to the nearest whole number because you need a whole number for calulating the norms
data["w1_age_round"] = np.round(data["w1_age"])

In [ ]:
# Create scaled scores for TMT-B that adjust for age
def map_raw_to_scaled(raw_score, age):
    """Maps raw TMT-B scores to scaled TMT-B scores based on age.

    Args:
        raw_score (int): Raw TMT-B score from filtered_df
        age (int): Age of participant
        TMTB (int): Raw TMT-B score from Neuronorma data
        Scale Score (int): Scaled TMT-B score from Neuronorma data

    Returns:
        TMTB_norm_age (int): Scaled TMT-B score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(int, age_range.split("-"))  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["TMTB"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None

data["tmtb_norm_age"] = data.apply(lambda row: map_raw_to_scaled(row["w1_tmt_b_raw"], row["w1_age_round"]), axis=1)

data[["id", "w1_age_round", "w1_tmt_b_raw", "tmtb_norm_age"]].head(10)

In [ ]:
# Create scaled scores for TMT-B that adjust now for education

# Readin Neuronorma data for education and format df
df_edu_data = pd.read_excel(
    "/Users/rachelmorse/superagers/Classification/neuronorma_education_data.xlsx",
    sheet_name="TMTB",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
TMTB_norm_list = []

for index, row in data.iterrows():
    tmtb_norm_age = row["tmtb_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    YoE = min(YoE, 20)

    TMTB_norm = df_edu_data.loc[tmtb_norm_age, YoE]
    TMTB_norm_list.append(TMTB_norm)  # Corrected here

data["tmtb_norm"] = TMTB_norm_list

data[["id", "YoE", "tmtb_norm_age", "tmtb_norm"]].head(10)

In [ ]:
# Calculate who is superager based off of RAVLT score

# Schmidt 1996 - age 16-29 RAVLT-Delayed recall is scoring 12+ (no data adjusted by sex)
data.loc[data["w1_delayed_recall_raw"] >= 12, "superager_ravlt"] = 1
data.loc[data["w1_delayed_recall_raw"] < 12, "superager_ravlt"] = 0

# Look at mean and standard deviation of RAVLT scores
mean_ravlt = data["w1_delayed_recall_raw"].mean()
std_dev_ravlt = data["w1_delayed_recall_raw"].std()
print(f"Mean RAVLT score: {mean_ravlt:.2f}")
print(f"Standard deviation of RAVLT score: {std_dev_ravlt:.2f}")

# Print mean age
mean_age = data["w1_age"].mean()
print(f"Mean age of participants: {mean_age:.2f}")

print("Number of superagers by RAVLT criteria:")
print(data[data["superager_ravlt"] == 1]["id"].count())

# Manually check that everything is running correctly
data[["id", "w1_delayed_recall_raw", "superager_ravlt"]].head(10)

With the scaled scores calculated, the mean is 10 and the SD is 3 for all variables. For the Neuronorma data, see [Peña-Casanova et al. (2009b)](https://pubmed.ncbi.nlm.nih.gov/19549723/) for more info. 

In [ ]:
# Create a superager variable that = 1 when superagers are above 1SD below the norm for TMT-B and meet the RAVLT criteria

# Define the variables and the lower bound
variables = ["tmtb_norm"]
lower_bound = 10 - 1 * 3  # Scaled score of 10 is the mean and SD is 3 for all variables

# Create new column 'superager' and initialize it to 0
data["superager"] = 0

# Update 'superager' to 1 for participants who score above the lower bound for TMT-B and have 1 for 'superager_RAVLT'
data.loc[(data[variables] >= lower_bound).all(axis=1) & (data["superager_ravlt"] == 1), "superager"] = 1

# Display a random sample of 10 rows
sample_df = data.sample(10)

print(data[data["superager"] == 1]["id"].count())

# Display the relevant columns
relevant_columns = ["id", "w1_age_round", "YoE", "superager"]
sample_df[relevant_columns]

In [ ]:
# Create a new df
clean_df = data

row_count = len(clean_df)
superager_count = clean_df["superager"].sum()
age_matched_controls = row_count - superager_count

print(" ")
print(f"Number of participants: {row_count}")
print(f"Number of superagers: {superager_count:.0f}")
print(f"Number of age-matched controls: {age_matched_controls:.0f}")

# Get basic info about superagers
superager_df = clean_df[clean_df["superager"] == 1]

average_age = superager_df["w1_age"].mean()
standard_deviation = superager_df["w1_age"].std()

print(f"Average superager age: {average_age:.2f}")
print(f"Standard deviation: {standard_deviation:.2f}")

# Get basic info about controls
controls_df = clean_df[clean_df["superager"] == 0]
average_age = controls_df["w1_age"].mean()
standard_deviation = controls_df["w1_age"].std()

print(f"Average control age: {average_age:.2f}")
print(f"Standard deviation: {standard_deviation:.2f}")

clean_df[["id", "superager", "w1_age", "YoE"]].head(15)

In [ ]:
# Filter the df to include only the needed variables
clean_df = clean_df[
    [
        "id",
        "w1_age",
        "YoE",
        "sex",
        "w1_delayed_recall_raw",
        "w1_tmt_b_raw",
        "w1_sem_fluency_raw",
        "w1_tmt_a_raw",
        "w1_direct_digits_raw",
        "w1_inverse_digits_raw",
        "w2_age",
        "w2_delayed_recall_raw",
        "w2_sem_fluency_raw",
        "w2_tmt_b_raw",
        "w2_tmt_a_raw",
        "w2_direct_digits_raw",
        "w2_inverse_digits_raw",
        "w1_crn30",
        "w2_crn30",
        "w1_cro30",
        "w2_cro30",
        "w1_ravlt_total",
        "w2_ravlt_total",
        "w1_crn",
        "w2_crn",
        "w1_cro",
        "w2_cro",
        "superager",
    ]
]

# Export this df to a csv to use for future analysis
clean_df.to_csv("/Users/rachelmorse/Documents/2023:2024/Data/Exported data/superager.csv", index=False)